<a href="https://colab.research.google.com/github/taichi0520-hub/my-research-project/blob/main/src/T_junction_simulation_simplified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1次Markov過程モデルに基づいたT字分岐のシミュレーションコード(CSVファイル出力まで)


### 使用方法


1.   「2. パラメータ設定」の部分の値を適宜変更して実行してください。それ以外の部分で記載されているn_cellsなどの値は、パラメータ設定の部分で明示的に指定されなかった場合のデフォルト値のため変更しなくて大丈夫です。

2.   コードの下部に「シミュレーションが完了し、結果を～～に保存しました。」と出力されたら、画面左のファイル(Google drive)の中にR(右折)とL(左折)が各セルに格納されたCSVファイルが保存されているはずなので、それを開いて解析してください。

*再度実行しても以前保存したファイルが上書きされないように、日時(yyyymmdd_hhmmss)をファイル名に入れ、ファイル名が重複しないようにしています。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
from datetime import datetime

# 1. 1次マルコフ過程に基づくシミュレーション関数
def simulate_t_junction(n_cells=20, alpha=0.0): #細胞数は適宜変更
    p_geo = 0.5
    results = [random.randint(0, 1)] # 先頭細胞 (0:Left, 1:Right)
    p_same = p_geo + alpha
    for i in range(1, n_cells):
        prev = results[-1]
        results.append(prev if random.random() < p_same else 1 - prev)
    return results

def get_cluster_sizes(sequence):
    if not sequence: return []
    clusters = []
    current_count = 1
    for i in range(1, len(sequence)):
        if sequence[i] == sequence[i-1]:
            current_count += 1
        else:
            clusters.append(current_count)
            current_count = 1
    clusters.append(current_count)
    return clusters

# 2. パラメータ設定（適宜変更して実行すること）
alphas = [0.0] # 相互作用パラメータ―αの値
n_trials = 20 # 試行回数
n_cells = 20 # 細胞数

print("--- シミュレーション結果のエクセル出力 ---\n")

for alpha in alphas:
    rows = []
    print(f"α = {alpha} でシミュレーション中...")
    for trial_id in range(1, n_trials + 1):
        seq_raw = simulate_t_junction(n_cells, alpha)
        clusters = get_cluster_sizes(seq_raw)
        mean_k = sum(clusters) / len(clusters)

        # 1行分のデータを作成 (パラメータは別のシートにするためここでは除外)
        row = {
            "Trial_ID": trial_id
        }

        # Cell_1からCell_20まで個別のキー（列）として格納
        for i, val in enumerate(seq_raw):
            row[f"Cell_{i+1}"] = "L" if val == 0 else "R"

        row["Cluster_Sizes"] = ",".join(map(str, clusters))
        row["Mean_Cluster_Size"] = round(mean_k, 2)
        rows.append(row)

    # 結果データのDataFrame化と列順序の整理
    df_results = pd.DataFrame(rows)
    cols = ["Trial_ID"] + [f"Cell_{i+1}" for i in range(n_cells)] + ["Cluster_Sizes", "Mean_Cluster_Size"]
    df_results = df_results[cols]

    # パラメータ設定をまとめたDataFrame
    df_params = pd.DataFrame({
        "Parameter": ["Alpha", "N_Cells", "N_Trials"],
        "Value": [alpha, n_cells, n_trials]
    })

    # タイムスタンプを使って一意のファイル名を生成 (Excel形式)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    excel_filename = f't_junction_sim_alpha_{alpha}_{timestamp}.xlsx'

    # ExcelWriterを使ってシートを分けて出力
    with pd.ExcelWriter(excel_filename) as writer:
        df_results.to_excel(writer, sheet_name="Results", index=False)
        df_params.to_excel(writer, sheet_name="Parameters", index=False)

    print(f"シミュレーションが完了し、結果を '{excel_filename}' に保存しました。")
